In [17]:
# import key libraries
library(emmeans)
library(dplyr)
library(afex)
library(effsize)

setwd("C:/Users/jacob/water_current_MA/data/mahalanobis_data")

# Load datasets
df_single <- read.csv("early_late_SINGLE_df.csv")
df_single$experiment <- factor("SINGLE")

df_dual <- read.csv("early_late_DUAL_df.csv")
df_dual$experiment <- factor("DUAL")

# Store data frames explicitly in a named list to avoid ls() collisions
data_list <- list(
  df_single = df_single,
  df_dual   = df_dual
)

# Function to format data types cleanly
format_datatypes <- function(df) {
    
    df$ppid_full <- factor(df$ppid_full)
    df$water_speed <- factor(df$water_speed)
    df$phase <- factor(df$phase)
    df$set_order <- factor(df$set_order)
    levels(df$set_order) <- c("group_1", "group_2")
    df$trial_set <- factor(df$trial_set)
      
    # Ensure consistent factor levels across all datasets
    df$target_id <- factor(df$target_id, levels = c("L60", "L30", "R30", "R60"))
      
    return(df)
}

# Apply formatting function to both data frames
data_list <- lapply(data_list, format_datatypes)

# Export formatted data frames back to the Global Environment
list2env(data_list, envir = .GlobalEnv)


<environment: R_GlobalEnv>

In [19]:
df_dual

ppid_full,set_order,target_id,water_speed,phase,trial_num,launch_angle,launch_speed,min_mahalanobis_distance,trial_set,experiment
<fct>,<fct>,<fct>,<fct>,<fct>,<int>,<dbl>,<dbl>,<dbl>,<fct>,<fct>
1_neg3,group_2,L60,-3,training_1,53,110.29240,1.893820,2.18921667,early,DUAL
1_neg3,group_2,L60,-3,training_1,55,44.36547,2.387203,0.26841689,early,DUAL
1_neg3,group_2,L60,-3,training_1,199,37.62541,3.947312,0.55865552,late,DUAL
1_neg3,group_2,L60,-3,training_1,201,36.78527,3.687697,0.38169800,late,DUAL
1_neg3,group_2,L30,-3,training_2,276,33.77943,2.923085,0.25321647,early,DUAL
1_neg3,group_2,L30,-3,training_2,278,50.93708,4.036107,1.49977590,early,DUAL
1_neg3,group_2,L30,-3,training_2,421,38.22177,4.062306,0.69933786,late,DUAL
1_neg3,group_2,L30,-3,training_2,424,47.50404,3.322672,0.41807124,late,DUAL
1_neg3,group_2,R30,-3,training_1,56,35.12555,1.952628,2.38975398,early,DUAL


# DUAL TARGET

## EARLY T1 vs LATE T1

In [21]:
# Isolate dual target experiment training phase 1
df_dual_t1 <- df_dual[df_dual$phase == 'training_1',]
speeds_list <- unique(df_dual_t1$water_speed)
set_order_list <- unique(df_dual_t1$set_order)

# reset
p_adj_inter <- 1.0
p_adj_main <- 1.0


for (speed in speeds_list) {
    
    results_storage <- list()
    raw_interaction_p_vals <- c()
    raw_trial_set_p_vals <- c()
    comparison_labels <- c()
        
    for (group in set_order_list) {
        
        df_subset <- df_dual_t1[df_dual_t1$water_speed == speed & df_dual_t1$set_order == group, ]
        df_subset <- droplevels(df_subset) 
        
        model_anova <- aov_ez(
            id = "ppid_full",              
            dv = "min_mahalanobis_distance",                  
            data = df_subset,                
            within = c("target_id", "trial_set"),
            fun_aggregate = mean 
        )
        
        label <- paste(speed, group)
        results_storage[[label]] <- model_anova
        
        raw_interaction_p_vals <- c(raw_interaction_p_vals, model_anova$anova_table["target_id:trial_set", "Pr(>F)"])
        raw_trial_set_p_vals <- c(raw_trial_set_p_vals, model_anova$anova_table["trial_set", "Pr(>F)"])
        comparison_labels <- c(comparison_labels, label)
        
        } 
            
            # Apply Holm corrections 
            adjusted_interaction_p_vals <- p.adjust(raw_interaction_p_vals, method = "holm")
            adjusted_trial_set_p_vals <- p.adjust(raw_trial_set_p_vals, method = "holm")
            
            names(adjusted_interaction_p_vals) <- comparison_labels
            names(adjusted_trial_set_p_vals) <- comparison_labels
        
            # POST-HOCS AND PRINTING
            for (label in comparison_labels) {
                
                model_anova <- results_storage[[label]]
                p_adj_inter <- adjusted_interaction_p_vals[label]
                p_adj_main <- adjusted_trial_set_p_vals[label]
                
                cat("\n========================================\n")
                cat("ANALYSIS FOR SPEED x GROUP:", label, "\n")
                cat("Holm-Adjusted Interaction p-value: ", p_adj_inter, "\n")
                cat("Holm-Adjusted trial_set Main Effect p-value: ", p_adj_main, "\n")
                cat("========================================\n")
                
                print(model_anova)
                
                # Conditional logic based on Holm-corrected results
                if (p_adj_inter <= 0.05) {
                    cat("\nSignificant Interaction. Running target-specific post-hocs:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set | target_id), adjust="holm"))
                    
                } else if (p_adj_main <= 0.05) {
                    cat("\nNon-Sig Interaction. Main Effect of trial_set:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set)))
                    
                } else {
                    cat("\nNeither effect survived Holm correction. Skipping post-hocs.\n")
                }
            } 
            
        }


ANALYSIS FOR SPEED x GROUP: -3 group_2 
Holm-Adjusted Interaction p-value:  0.03569423 
Holm-Adjusted trial_set Main Effect p-value:  2.570485e-07 
Anova Table (Type 3 tests)

Response: min_mahalanobis_distance
               Effect    df  MSE         F  ges p.value
1           target_id 1, 19 0.23 46.88 *** .242   <.001
2           trial_set 1, 19 0.27 66.32 *** .352   <.001
3 target_id:trial_set 1, 19 0.25    5.78 * .042    .027
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '+' 0.1 ' ' 1

Significant Interaction. Running target-specific post-hocs:
target_id = L60:
 contrast     estimate    SE df t.ratio p.value
 early - late    0.681 0.126 19   5.414  <.0001

target_id = R30:
 contrast     estimate    SE df t.ratio p.value
 early - late    1.224 0.192 19   6.363  <.0001


ANALYSIS FOR SPEED x GROUP: -3 group_1 
Holm-Adjusted Interaction p-value:  0.03569423 
Holm-Adjusted trial_set Main Effect p-value:  2.570485e-07 
Anova Table (Type 3 tests)

Response: min_mahalanobis_dista

## EARLY T2 vs LATE T2

In [33]:
# Isolate dual target experiment training phase 1
df_dual_t2 <- df_dual[df_dual$phase == 'training_2',]
speeds_list <- unique(df_dual_t2$water_speed)
set_order_list <- unique(df_dual_t2$set_order)

# reset
p_adj_inter <- 1.0
p_adj_main <- 1.0


for (speed in speeds_list) {
    
    results_storage <- list()
    raw_interaction_p_vals <- c()
    raw_trial_set_p_vals <- c()
    comparison_labels <- c()
        
    for (group in set_order_list) {
        
        df_subset <- df_dual_t2[df_dual_t2$water_speed == speed & df_dual_t2$set_order == group, ]
        df_subset <- droplevels(df_subset) 
        
        model_anova <- aov_ez(
            id = "ppid_full",              
            dv = "min_mahalanobis_distance",                  
            data = df_subset,                
            within = c("target_id", "trial_set"),
            fun_aggregate = mean 
        )
        
        label <- paste(speed, group)
        results_storage[[label]] <- model_anova
        
        raw_interaction_p_vals <- c(raw_interaction_p_vals, model_anova$anova_table["target_id:trial_set", "Pr(>F)"])
        raw_trial_set_p_vals <- c(raw_trial_set_p_vals, model_anova$anova_table["trial_set", "Pr(>F)"])
        comparison_labels <- c(comparison_labels, label)
        
        } 
            
            # Apply Holm corrections 
            adjusted_interaction_p_vals <- p.adjust(raw_interaction_p_vals, method = "holm")
            adjusted_trial_set_p_vals <- p.adjust(raw_trial_set_p_vals, method = "holm")
            
            names(adjusted_interaction_p_vals) <- comparison_labels
            names(adjusted_trial_set_p_vals) <- comparison_labels
        
            # POST-HOCS AND PRINTING
            for (label in comparison_labels) {
                
                model_anova <- results_storage[[label]]
                p_adj_inter <- adjusted_interaction_p_vals[label]
                p_adj_main <- adjusted_trial_set_p_vals[label]
                
                cat("\n========================================\n")
                cat("ANALYSIS FOR SPEED x GROUP:", label, "\n")
                cat("Holm-Adjusted Interaction p-value: ", p_adj_inter, "\n")
                cat("Holm-Adjusted trial_set Main Effect p-value: ", p_adj_main, "\n")
                cat("========================================\n")
                
                print(model_anova)
                
                # Conditional logic based on Holm-corrected results
                if (p_adj_inter <= 0.05) {
                    cat("\nSignificant Interaction. Running target-specific post-hocs:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set | target_id), adjust="holm"))
                    
                } else if (p_adj_main <= 0.05) {
                    cat("\nNon-Sig Interaction. Main Effect of trial_set:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set)))
                    
                } else {
                    cat("\nNeither effect survived Holm correction. Skipping post-hocs.\n")
                }
            } 
            
        }


ANALYSIS FOR SPEED x GROUP: -3 group_2 
Holm-Adjusted Interaction p-value:  0.6870235 
Holm-Adjusted trial_set Main Effect p-value:  0.01441427 
Anova Table (Type 3 tests)

Response: min_mahalanobis_distance
               Effect    df  MSE       F  ges p.value
1           target_id 1, 19 0.36  7.75 * .118    .012
2           trial_set 1, 19 0.27 9.06 ** .104    .007
3 target_id:trial_set 1, 19 0.13    0.17 .001    .687
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '+' 0.1 ' ' 1

Non-Sig Interaction. Main Effect of trial_set:
 contrast     estimate    SE df t.ratio p.value
 early - late    0.348 0.116 19   3.010  0.0072

Results are averaged over the levels of: target_id 

ANALYSIS FOR SPEED x GROUP: -3 group_1 
Holm-Adjusted Interaction p-value:  0.6138664 
Holm-Adjusted trial_set Main Effect p-value:  0.1490126 
Anova Table (Type 3 tests)

Response: min_mahalanobis_distance
               Effect    df  MSE      F  ges p.value
1           target_id 1, 24 0.30 5.21 * .068    .0

## TRANSFER

In [27]:

# Filter for Transfer (Early trials only)
df_dual_transfer <- df_dual[df_dual$trial_set == 'early',]
df_dual_transfer$phase <- factor(df_dual_transfer$phase, levels = c("training_1", "training_2"))

speeds_list <- unique(df_dual_transfer$water_speed)
target_list <- unique(df_dual_transfer$target_id)



for (s in speeds_list) {
    print(s)

    p_vals <- c()
    
    for (t in target_list) {
        print(t)
  
          cat("\n--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET:", s, t, "---\n")
          
          df_subset <- df_dual_transfer[df_dual_transfer$water_speed == s & df_dual_transfer$target_id == t,]
          df_subset <- droplevels(df_subset) 
          
          res <- t.test(min_mahalanobis_distance ~ phase, data = df_subset, alternative = "greater")
          print(res)

          # show sd
          print('sd')
          print(tapply(df_subset$min_mahalanobis_distance, df_subset$phase, sd, na.rm = TRUE))
        
          # effect sizes
          print(cohen.d(min_mahalanobis_distance ~ phase, data = df_subset))
        
          # Grab p-value & store
          p_vals <- c(p_vals, res$p.value)

    }
    
    # show p-vals
    cat(' Raw p-vals for',s,'water speed:\n')
    print(p_vals)
    
    # adjusted p-vals
    cat(' Adjusted p-vals for',s,'water speed:\n')
    adj_p <- p.adjust(p_vals, method = "holm")
    print(adj_p)
}


[1] "-3"
[1] "L60"

--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET: -3 L60 ---

	Welch Two Sample t-test

data:  min_mahalanobis_distance by phase
t = 4.4153, df = 51.912, p-value = 2.568e-05
alternative hypothesis: true difference in means between group training_1 and group training_2 is greater than 0
95 percent confidence interval:
 0.3873179       Inf
sample estimates:
mean in group training_1 mean in group training_2 
               0.9598388                0.3358330 

[1] "sd"
training_1 training_2 
 0.8276394  0.3774234 

Cohen's d

d estimate: 1.008442 (large)
95 percent confidence interval:
   lower    upper 
0.561192 1.455691 

[1] "L30"

--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET: -3 L30 ---

	Welch Two Sample t-test

data:  min_mahalanobis_distance by phase
t = 1.4065, df = 83.173, p-value = 0.08166
alternative hypothesis: true difference in means between group training_1 and group training_2 is greater than 0
95 percent confidence interval:
 -0.0326

In [74]:
# Check unique values and counts of phase in df_subset
table(df_subset$phase, useNA = "ifany")


training_1 
        40 

# SINGLE TARGET

In [52]:
# Isolate dual target experiment training phase 1
df_single <- combined_df[combined_df$experiment == "SINGLE",]
df_single <- df_single[df_single$phase == 'training_1',]
speeds_list <- unique(df_single$water_speed)
set_order_list <- unique(df_single$set_order)
target_list <- unique(df_single$target_id)


# reset
p_adj_inter <- 1.0
p_adj_main <- 1.0

for (speed in speeds_list) {
    
    results_storage <- list()
    raw_interaction_p_vals <- c()
    raw_trial_set_p_vals <- c()
    comparison_labels <- c()
        
    for (target in target_list) {
        
        df_subset <- df_single[df_single$water_speed == speed & df_single$target_id == target, ]
        df_subset <- droplevels(df_subset) 
        
        model_anova <- aov_ez(
            id = "ppid_full",              
            dv = "min_mahalanobis_distance",                  
            data = df_subset,                
            within = "trial_set",
            fun_aggregate = mean 
        )
        
        label <- paste(speed, target)
        results_storage[[label]] <- model_anova
        
        # raw_interaction_p_vals <- c(raw_interaction_p_vals, model_anova$anova_table["target_id:trial_set", "Pr(>F)"])
        raw_trial_set_p_vals <- c(raw_trial_set_p_vals, model_anova$anova_table["trial_set", "Pr(>F)"])
        comparison_labels <- c(comparison_labels, label)
        
        } 
            
            # Apply Holm corrections 
            # adjusted_interaction_p_vals <- p.adjust(raw_interaction_p_vals, method = "holm")
            adjusted_trial_set_p_vals <- p.adjust(raw_trial_set_p_vals, method = "holm")
            
            # names(adjusted_interaction_p_vals) <- comparison_labels
            names(adjusted_trial_set_p_vals) <- comparison_labels
        
            # POST-HOCS AND PRINTING
            for (label in comparison_labels) {
                
                model_anova <- results_storage[[label]]
                # p_adj_inter <- adjusted_interaction_p_vals[label]
                p_adj_main <- adjusted_trial_set_p_vals[label]
                
                cat("\n========================================\n")
                cat("ANALYSIS FOR SPEED x GROUP:", label, "\n")
                # cat("Holm-Adjusted Interaction p-value: ", p_adj_inter, "\n")
                cat("Holm-Adjusted trial_set Main Effect p-value: ", p_adj_main, "\n")
                cat("========================================\n")
                
                print(model_anova)
                
                # Conditional logic based on Holm-corrected results
                if (p_adj_inter <= 0.05) {
                    cat("\nSignificant Interaction. Running target-specific post-hocs:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set), adjust="holm"))
                    
                } else if (p_adj_main <= 0.05) {
                    cat("\nNon-Sig Interaction. Main Effect of trial_set:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set)))
                    
                } else {
                    cat("\nNeither effect survived Holm correction. Skipping post-hocs.\n")
                }
            } 
            
        }


ANALYSIS FOR SPEED x GROUP: -3 R60 
Holm-Adjusted trial_set Main Effect p-value:  4.700784e-10 
Anova Table (Type 3 tests)

Response: min_mahalanobis_distance
     Effect    df  MSE          F  ges p.value
1 trial_set 1, 19 0.23 145.67 *** .691   <.001
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '+' 0.1 ' ' 1

Non-Sig Interaction. Main Effect of trial_set:
 contrast     estimate   SE df t.ratio p.value
 early - late     1.81 0.15 19  12.069  <.0001


ANALYSIS FOR SPEED x GROUP: -3 L60 
Holm-Adjusted trial_set Main Effect p-value:  1.925487e-07 
Anova Table (Type 3 tests)

Response: min_mahalanobis_distance
     Effect    df  MSE         F  ges p.value
1 trial_set 1, 15 0.14 81.23 *** .661   <.001
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '+' 0.1 ' ' 1

Non-Sig Interaction. Main Effect of trial_set:
 contrast     estimate    SE df t.ratio p.value
 early - late     1.21 0.134 15   9.013  <.0001

